# Experiment 1: Hardening Strategy Comparison

Progressive stacking with 5 hardening strategies on Sudoku-Extreme 9x9.

**Prerequisites**: Runtime > Change runtime type > **GPU (T4)**

| Strategy | Hardening | Gear analogy |
|----------|-----------|-------------|
| gradual (SoftGear) | Per-layer LR decay | Gradual hardening |
| none | All layers same LR | All gears stay soft |
| freeze | Existing layers LR=0 | Existing gears turn to stone |
| binary | Old 0.4x / new 1.0x | Uniform hardening |
| from_scratch | No progressive stacking | All gears mounted at once |

## Setup

In [ ]:
!pip install -q git+https://github.com/byExist/softgear.git

In [ ]:
!softgear download sudoku9

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available! Change runtime type."
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# type: ignore
from google.colab import drive

drive.mount("/content/drive")
CKPT_ROOT = "/content/drive/MyDrive/softgear/checkpoints"

In [ ]:
COMMON = f"--task sudoku9 --num-gears 7 --hidden-dim 128 --ffn-dim 512 --num-heads 4 --patience 5 --max-total-steps 40000 --max-samples 50000 --seed 42"

In [ ]:
# gradual (SoftGear)
!softgear train --hardening gradual --checkpoint-dir {CKPT_ROOT}/exp1-gradual $COMMON

In [ ]:
# none (no protection)
!softgear train --hardening none --checkpoint-dir {CKPT_ROOT}/exp1-none $COMMON

In [ ]:
# freeze (full freeze)
!softgear train --hardening freeze --checkpoint-dir {CKPT_ROOT}/exp1-freeze $COMMON

In [ ]:
# binary (binary LR)
!softgear train --hardening binary --checkpoint-dir {CKPT_ROOT}/exp1-binary $COMMON

In [ ]:
# from_scratch (all gears at once)
!softgear train --hardening from_scratch --checkpoint-dir {CKPT_ROOT}/exp1-scratch $COMMON

## Results

In [ ]:
import torch
import pandas as pd
from typing import Any
from softgear.tasks.registry import get_task
from softgear.config import DataConfig, ModelConfig

task = get_task("sudoku9")
data_cfg = DataConfig(path="data/sudoku-extreme", batch_size=64, num_workers=2, max_samples=None, curriculum=False)
_, val_loader = task.build_loaders(data_cfg)
device = torch.device("cuda")

strategies = {
    "gradual": f"{CKPT_ROOT}/exp1-gradual",
    "none": f"{CKPT_ROOT}/exp1-none",
    "freeze": f"{CKPT_ROOT}/exp1-freeze",
    "binary": f"{CKPT_ROOT}/exp1-binary",
    "from_scratch": f"{CKPT_ROOT}/exp1-scratch",
}

results: dict[str, Any] = {}
for name, ckpt_dir in strategies.items():
    ckpt = torch.load(f"{ckpt_dir}/best.pt", map_location=device, weights_only=False)
    cfg = ckpt["config"]

    model_cfg = ModelConfig(**cfg["model"])
    model = task.build_model(model_cfg)
    task.mount_all_gears(model, model_cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    all_preds: list[torch.Tensor] = []
    all_targets: list[torch.Tensor] = []
    all_inputs: list[torch.Tensor] = []
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch[0].to(device), batch[1].to(device)
            preds = task.predict_fn(model(x).logits)
            all_preds.append(preds)
            all_targets.append(y)
            all_inputs.append(x)

    metrics = task.metrics_fn(torch.cat(all_preds), torch.cat(all_targets), torch.cat(all_inputs))
    results[name] = {
        "phase": ckpt["phase"],
        "global_step": ckpt.get("global_step", "?"),
        "val_loss": ckpt["best_val_loss"],
        **metrics,
    }

df = pd.DataFrame(results).T
df.index.name = "strategy"
df